In [1]:
import pandas as pd
import numpy as np
import optuna
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

X_train = pd.read_csv('../data/processed/splits/X_train.csv')
X_test  = pd.read_csv('../data/processed/splits/X_test.csv')
y_train = pd.read_csv('../data/processed/splits/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/processed/splits/y_test.csv').squeeze()

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")

X_train: (452296, 55)
X_test:  (61503, 55)


In [2]:
def objective(trial):
    params = {
        'n_estimators':    trial.suggest_int('n_estimators', 100, 500),
        'max_depth':       trial.suggest_int('max_depth', 3, 8),
        'learning_rate':   trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves':      trial.suggest_int('num_leaves', 20, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample':       trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':       trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':      trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'class_weight': 'balanced',
        'random_state': 42,
        'verbose': -1
    }

    # 3-fold cross validation on training data
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    auc_scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]
        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = LGBMClassifier(**params)
        model.fit(X_tr, y_tr)
        y_prob = model.predict_proba(X_val)[:, 1]
        auc_scores.append(roc_auc_score(y_val, y_prob))

    return np.mean(auc_scores)

In [3]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"\nBest trial AUC:   {study.best_value:.4f}")
print(f"Best parameters:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

  0%|          | 0/30 [00:00<?, ?it/s]


Best trial AUC:   0.9784
Best parameters:
  n_estimators: 499
  max_depth: 8
  learning_rate: 0.07270620979682454
  num_leaves: 145
  min_child_samples: 98
  subsample: 0.598911188091524
  colsample_bytree: 0.5045731985056197
  reg_alpha: 0.0002389578940077299
  reg_lambda: 5.6224563011461495


In [4]:
best_params = study.best_params
best_params['class_weight'] = 'balanced'
best_params['random_state'] = 42
best_params['verbose'] = -1

final_model = LGBMClassifier(**best_params)
final_model.fit(X_train, y_train)

y_prob_tuned = final_model.predict_proba(X_test)[:, 1]
auc_tuned = roc_auc_score(y_test, y_prob_tuned)

print(f"Tuned LightGBM ROC-AUC: {auc_tuned:.4f}")
print(f"Baseline LightGBM ROC-AUC: 0.7012")
print(f"Improvement: +{(auc_tuned - 0.7012):.4f}")

Tuned LightGBM ROC-AUC: 0.7526
Baseline LightGBM ROC-AUC: 0.7012
Improvement: +0.0514


In [6]:
import joblib
import os

os.makedirs('../models', exist_ok=True)

# Save final tuned model
joblib.dump(final_model, '../models/lgbm_best_model.pkl')

print("Model saved to models/lgbm_best_model.pkl")

Model saved to models/lgbm_best_model.pkl
